In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import random

# Import từ các file của bạn
from model import get_model
import config
from metrics import compute_eer, compute_mindcf # Tận dụng hàm của nhóm bạn

# ============================================================================
# CONFIGURATION
# ============================================================================
PTM_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding"
HC_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after extract feature"
WEIGHTS_DIR = r"./checkpoints"
GT_CSV = "test_list_gt.csv"

PTMS = ["wavlm", "hubert", "wav2vec2"]
HANDCRAFTEDS = ["mfcc_only", "mfbe_only", "pitch_only", "mfcc_pitch", "mfbe_pitch"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cố định seed như code của nhóm bạn
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

@torch.no_grad()
def run_benchmark(ptm_name, hc_mode, test_pairs):
    print(f"\n>>> [BENCHMARK] PTM: {ptm_name.upper()} | Handcrafted: {hc_mode.upper()}")
    
    # 1. Load Features
    ptm_feat_path = os.path.join(PTM_DIR, f"all_embeddings_{ptm_name}.pt")
    hc_feat_path = os.path.join(HC_DIR, f"all_features_{hc_mode.replace('only', 'Only').replace('pitch', 'Pitch')}.pt")
    
    if not (os.path.exists(ptm_feat_path) and os.path.exists(hc_feat_path)):
        print(f"Skipping: Thiếu file feature.")
        return None

    ptm_data = torch.load(ptm_feat_path, map_location='cpu')
    hc_data = torch.load(hc_feat_path, map_location='cpu')

    # 2. Khởi tạo Model & Load Weights
    model = get_model(num_speakers=config.NUM_CLASSES if hasattr(config, 'NUM_CLASSES') else 1000, 
                      mode=3, 
                      feature_mode=hc_mode, 
                      device=DEVICE)
    
    weight_file = os.path.join(WEIGHTS_DIR, f"best_model_{ptm_name}_{hc_mode}.pth")
    if not os.path.exists(weight_file):
        print(f"Skipping: Không thấy file trọng số {weight_file}")
        return None
        
    checkpoint = torch.load(weight_file, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # 3. Trích xuất Embeddings và Chuẩn hóa (Theo logic code mới của nhóm)
    scores = []
    labels = []

    for _, row in tqdm(test_pairs.iterrows(), total=len(test_pairs), desc="Scoring"):
        k1, k2 = row['key1'], row['key2']
        if k1 not in ptm_data or k2 not in ptm_data or k1 not in hc_data or k2 not in hc_data:
            continue

        # Đẩy lên GPU và ép kiểu
        p1_ptm, p1_hc = ptm_data[k1].unsqueeze(0).to(DEVICE).to(torch.float32), hc_data[k1].unsqueeze(0).to(DEVICE).to(torch.float32)
        p2_ptm, p2_hc = ptm_data[k2].unsqueeze(0).to(DEVICE).to(torch.float32), hc_data[k2].unsqueeze(0).to(DEVICE).to(torch.float32)

        # Forward
        emb1 = model(p1_ptm, p1_hc, get_embedding=True)
        emb2 = model(p2_ptm, p2_hc, get_embedding=True)

        # L2 Normalization (Tích hợp từ code nhóm bạn)
        emb1 = F.normalize(emb1, p=2, dim=1)
        emb2 = F.normalize(emb2, p=2, dim=1)

        # Tính Score (Dot product lúc này tương đương Cosine similarity)
        score = torch.sum(emb1 * emb2, dim=1)
        scores.append(score.item())
        labels.append(int(row['label']))

    if len(scores) == 0: return None
    
    # 4. Tính toán Metrics (Tích hợp MinDCF từ code nhóm bạn)
    scores = np.array(scores)
    labels = np.array(labels)
    
    eer, _ = compute_eer(labels, scores)
    min_dcf, _ = compute_mindcf(labels, scores, p_target=0.05)
    
    print(f"Result -> EER: {eer*100:.2f}% | MinDCF: {min_dcf:.4f}")
    return {"EER": eer*100, "MinDCF": min_dcf}

if __name__ == "__main__":
    # Tiền xử lý CSV
    df = pd.read_csv(GT_CSV, header=None, names=['raw'])
    df[['label', 'path1', 'path2']] = df['raw'].str.strip('"').str.split('\t', expand=True)
    df['key1'] = df['path1'].apply(lambda x: x.split('/')[-1])
    df['key2'] = df['path2'].apply(lambda x: x.split('/')[-1])

    all_results = []

    for ptm in PTMS:
        for hc in HANDCRAFTEDS:
            res = run_benchmark(ptm, hc, df)
            if res:
                all_results.append({
                    "PTM": ptm, 
                    "Handcrafted": hc, 
                    "EER (%)": round(res["EER"], 2),
                    "MinDCF": round(res["MinDCF"], 4)
                })
                torch.cuda.empty_cache()

    # Xuất báo cáo
    if all_results:
        final_df = pd.DataFrame(all_results)
        print("\n" + "="*60)
        print("BẢNG TỔNG HỢP ĐÁNH GIÁ (EER & MinDCF)")
        print("="*60)
        print(final_df)
        final_df.to_csv("final_inference_report.csv", index=False)